# TravelMind Coding Exercise 3: Advanced
### Give after Day 6 is fully taught (swarm, graph, composition)

Two hard problems land the same night. A monsoon grounds 40 flights and nobody can say in what order the work should happen. Sofia needs involuntary rebookings that a regulator can audit later. Dev, the on-call SRE, gets paged if either one runs up an open-ended bill.

You are at the top of the ladder now. The skill on show is not building the biggest system. It is bounding the one you build.

**What is inside:**
- Setup you run once (model, mock tools, deterministic ops for identity, compensation, policy, audit, and a multi-agent meter).
- Eleven tasks: MCQ, choose-an-option, fill-the-blank, a node-type table, two flowcharts to complete, spot-the-errors, debug-and-fix, predict-execution-order, and two implement-from-a-flow builds (an auditable graph and a composition).

**Rules:** anchor booking PNR `JX48Q2`, surname `Rao`, Gold, `BLR-DEL` cancelled by the airline. Run setup first. Placeholders are `____` or `# TODO`.

## How to run
- **VS Code:** Python 3.10+ kernel, boto3 with Bedrock access to Haiku 4.5 in `us-east-1`, run top to bottom.
- **Colab:** upload, run install, set AWS env vars and `AWS_REGION`, run top to bottom.

In [ ]:
%pip install -q strands-agents strands-agents-tools nest_asyncio matplotlib

In [ ]:
import os, time, json, contextlib
import nest_asyncio
nest_asyncio.apply()

from strands import Agent, tool
from strands.models import BedrockModel

REGION   = os.environ.get("AWS_REGION", "us-east-1")
HAIKU_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
haiku    = BedrockModel(model_id=HAIKU_ID, region_name=REGION, temperature=0.3)
haiku_t0 = BedrockModel(model_id=HAIKU_ID, region_name=REGION, temperature=0)   # deterministic gate agents
print("Models ready.")

In [ ]:
# Base mock airline operations.
_PNRS = {"JX48Q2": {"surname": "Rao", "passenger_id": "P-100294", "loyalty_tier": "Gold",
    "segments": [{"flight": "6E-317", "from": "BLR", "to": "DEL", "dep": "2026-07-05T08:10",
                  "status": "CANCELLED", "fare_basis": "TA21R"},
                 {"flight": "6E-512", "from": "DEL", "to": "BOM", "dep": "2026-07-05T13:40",
                  "status": "ON TIME", "fare_basis": "TA21R"}],
    "ancillaries": {"seat": "14C (paid)", "bags": 1}}}
_FARE_RULES = {"TA21R": {"refundable": False, "change_fee": 3000, "currency": "INR",
    "notes": "Non-refundable on voluntary change. Airline-caused (involuntary) changes waive fees."}}


@tool
def get_pnr(record_locator: str, surname: str) -> str:
    """Look up a booking by record locator and surname.

    Args:
        record_locator: 6-character PNR.
        surname: Passenger surname.
    Returns:
        JSON string of the booking, or an error.
    """
    rec = _PNRS.get(record_locator.upper())
    if not rec or rec["surname"].lower() != surname.lower():
        return json.dumps({"error": "PNR not found or surname mismatch"})
    return json.dumps(rec)


@tool
def get_fare_rules(fare_basis: str) -> str:
    """Return fare rules for a fare basis code.

    Args:
        fare_basis: Fare basis code.
    Returns:
        JSON string of fare rules.
    """
    return json.dumps(_FARE_RULES.get(fare_basis.upper(), {"error": "unknown fare basis"}))


@tool
def search_reaccommodation(origin: str, destination: str, after: str) -> str:
    """Find alternative flights after a disruption.

    Args:
        origin: Origin airport code.
        destination: Destination airport code.
        after: ISO datetime lower bound.
    Returns:
        JSON string list of candidate flights.
    """
    return json.dumps([{"flight": "6E-333", "from": origin, "to": destination, "dep": "2026-07-05T11:20", "seats": 4},
                       {"flight": "AI-809", "from": origin, "to": destination, "dep": "2026-07-05T15:05", "seats": 9}])


@tool
def get_loyalty(passenger_id: str) -> str:
    """Return loyalty tier and benefits.

    Args:
        passenger_id: Internal passenger id.
    Returns:
        JSON string of tier and benefits.
    """
    return json.dumps({"passenger_id": passenger_id, "tier": "Gold",
                       "benefits": ["priority rebooking", "waived change fee on same-day", "lounge access"]})


@tool
def check_refund_eligibility(record_locator: str, reason: str) -> str:
    """Assess refund eligibility from fare rules and reason.

    Args:
        record_locator: PNR.
        reason: Free-text reason.
    Returns:
        JSON string with eligibility and the controlling rule.
    """
    rec = _PNRS.get(record_locator.upper(), {})
    if not rec:
        return json.dumps({"error": "PNR not found"})
    rules = _FARE_RULES.get(rec["segments"][0]["fare_basis"], {})
    airline_caused = any(k in reason.lower() for k in ["cancel", "delay", "airline", "irrops"])
    return json.dumps({"eligible": airline_caused or rules.get("refundable", False),
                       "airline_caused": airline_caused, "rule": rules.get("notes", "")})

print("Base tools ready.")

In [ ]:
# Deterministic operations for an auditable flow: identity, compensation, policy, audit.
AUDIT_LOG = []


@tool
def validate_identity(record_locator: str, surname: str) -> str:
    """Hard identity check against ground-truth booking data.

    Args:
        record_locator: PNR.
        surname: Passenger surname.
    Returns:
        JSON {"verified": bool, "passenger_id": str|null}.
    """
    rec = _PNRS.get(record_locator.upper())
    ok = bool(rec) and rec["surname"].lower() == surname.lower()
    return json.dumps({"verified": ok, "passenger_id": rec["passenger_id"] if ok else None})


@tool
def compute_compensation(disruption_type: str, loyalty_tier: str) -> str:
    """Duty-of-care compensation by disruption type and tier.

    Args:
        disruption_type: e.g. 'cancellation' or 'delay'.
        loyalty_tier: e.g. 'Gold'.
    Returns:
        JSON with voucher value (INR) and care items.
    """
    base = {"cancellation": 5000, "delay": 2500}.get(disruption_type.lower(), 0)
    bump = {"Gold": 1.5, "Silver": 1.2}.get(loyalty_tier, 1.0)
    care = ["meal voucher", "priority rebooking"] + (["hotel if overnight"] if disruption_type.lower() == "cancellation" else [])
    return json.dumps({"voucher_inr": int(base * bump), "care": care, "tier_applied": loyalty_tier})


@tool
def apply_policy_gate(record_locator: str, disruption_reason: str) -> str:
    """Deterministic policy gate using ground-truth data.

    Args:
        record_locator: PNR.
        disruption_reason: Free-text reason.
    Returns:
        JSON {"pass": bool, "reasons": [...]}.
    """
    rec = _PNRS.get(record_locator.upper(), {})
    if not rec:
        return json.dumps({"pass": False, "reasons": ["PNR not found"]})
    rules = _FARE_RULES.get(rec["segments"][0]["fare_basis"], {})
    airline_caused = any(k in disruption_reason.lower() for k in ["cancel", "delay", "airline", "irrops"])
    passed = airline_caused or bool(rules.get("refundable"))
    reasons = [] if passed else ["Voluntary change on a non-refundable fare cannot be auto-approved."]
    return json.dumps({"pass": passed, "reasons": reasons})


@tool
def write_audit(event: str) -> str:
    """Append an immutable audit record.

    Args:
        event: Description of the decision to log.
    Returns:
        JSON confirmation with the sequence number.
    """
    seq = len(AUDIT_LOG)
    AUDIT_LOG.append({"seq": seq, "event": event})
    return json.dumps({"logged": True, "seq": seq})

print("Deterministic ops ready.")

In [ ]:
# Multi-agent meter (reads Graph and Swarm usage).
PRICES = {"haiku": (1.00, 5.00)}   # illustrative; verify before quoting
_LEDGER, RESULTS = [], {}

def usage_of(result):
    u = getattr(getattr(result, "metrics", None), "accumulated_usage", None)
    if u is None:
        u = getattr(result, "accumulated_usage", None) or {}
    return {"input": u.get("inputTokens", 0) or u.get("input_tokens", 0),
            "output": u.get("outputTokens", 0) or u.get("output_tokens", 0)}

def _nodes_run(result):
    seq = getattr(result, "execution_order", None) or getattr(result, "node_history", None) or []
    return [getattr(n, "node_id", None) or getattr(n, "id", None) or str(n) for n in seq]

def final_text(result):
    ids = _nodes_run(result)
    if ids:
        try:
            return str(result.results[ids[-1]].result)
        except Exception:
            pass
    return str(result)

def metered_multi(orchestrator, prompt, tier="haiku"):
    res = orchestrator(prompt)
    _LEDGER.append((tier, usage_of(res), max(1, len(_nodes_run(res)))))
    return final_text(res), res

@contextlib.contextmanager
def meter(label):
    _LEDGER.clear(); t0 = time.time()
    try:
        yield
    finally:
        lat = time.time() - t0; cost = tin = tout = n = 0
        for tier, u, c in _LEDGER:
            pin, pout = PRICES[tier]
            cost += u["input"]/1e6*pin + u["output"]/1e6*pout; tin += u["input"]; tout += u["output"]; n += c
        RESULTS[label] = {"latency_s": round(lat, 2), "tokens": tin + tout, "calls": n, "cost_usd": round(cost, 6)}
        print(f"[{label}] nodes={n} latency={round(lat,2)}s tokens={tin+tout} cost=${round(cost,6):.6f}")

print("Meter ready.")

## Task 1: MCQ, match the sub-scenario

- **S-A, the storm:** re-accommodation, waivers, duty-of-care, and comms all interact, order unknown.
- **S-B, the auditor:** identity by hard rule, every decision logged, identical path each run.

Set your answers.

In [ ]:
# a) graph   b) swarm   c) routing   d) chaining
ANSWER_SA = "?"   # S-A the storm
ANSWER_SB = "?"   # S-B the auditor
print(ANSWER_SA, ANSWER_SB)

## Task 2: Fill the blank, build the IRROPS swarm

A swarm is peers with shared memory and no boss. Each agent gets an auto-injected `handoff_to_agent` tool. Complete the constructor: the entry agent (as an object), and the ping-pong guards.

In [ ]:
from strands.multiagent import Swarm

sw_reaccom = Agent(model=haiku, name="reaccom_specialist",
    system_prompt="Find alternative flights. Hand off to fare_specialist for waivers, compensation_specialist for care, comms_specialist to write up.",
    tools=[get_pnr, search_reaccommodation])
sw_fare = Agent(model=haiku, name="fare_specialist",
    system_prompt="Confirm fare rules and involuntary status. Hand off to reaccom_specialist, compensation_specialist, or comms_specialist.",
    tools=[get_pnr, get_fare_rules])
sw_comp = Agent(model=haiku, name="compensation_specialist",
    system_prompt="Compute duty-of-care compensation. Hand off to comms_specialist when ready.",
    tools=[get_pnr, get_loyalty, compute_compensation])
sw_comms = Agent(model=haiku, name="comms_specialist",
    system_prompt="Write the final customer message from what the team gathered. This ends the task.")

irrops_swarm = Swarm(
    [sw_reaccom, sw_fare, sw_comp, sw_comms],
    entry_point=____,                            # TODO: which agent starts (pass the object)
    max_handoffs=12,
    max_iterations=12,
    execution_timeout=600.0,
    node_timeout=180.0,
    repetitive_handoff_detection_window=____,    # TODO: look-back window for ping-pong
    repetitive_handoff_min_unique_agents=____,   # TODO: min distinct agents in that window
)
print("swarm built")

## Task 3: Debug, this swarm is broken four ways

Run it. It has **four** defects: a wrong import, a missing setting on one agent, a wrong-typed `entry_point`, and missing guards. Fix all four, then re-run.

In [ ]:
from strands.multi_agent import Swarm as BadSwarm   # defect 1

d_reaccom = Agent(model=haiku, system_prompt="Find alternative flights. Hand off when done.")   # defect 2
d_fare    = Agent(model=haiku, name="fare_agent", system_prompt="Confirm fare rules and waivers.")
d_comms   = Agent(model=haiku, name="comms_agent", system_prompt="Write the customer message.")

bad_swarm = BadSwarm(
    [d_reaccom, d_fare, d_comms],
    entry_point="reaccom",   # defect 3
    # defect 4: no guards, no timeouts
)
print("if this printed, you fixed the import at least")

## Task 4: Spot the errors in the graph

`validate` fans out to `reaccom` and `comp`; both feed `gate`.

```python
b.add_edge("validate", "reaccom")
b.add_edge("validate", "comp")
b.add_edge("reaccom", "gate")
b.add_edge("comp",    "gate")

def policy_passed(state):
    r = state.results.get("policygate")     # the node was added as "gate"
    return bool(r) and "policy pass" in str(r.result).lower()
```

Two problems. Record them.

- Error 1 (the diamond, under Python OR semantics): ________
- Error 2 (the condition): ________

In [ ]:
DIAMOND_BUG   = "____"   # TODO: what goes wrong when reaccom finishes before comp
CONDITION_BUG = "____"   # TODO: what is wrong in policy_passed
print(DIAMOND_BUG); print(CONDITION_BUG)

## Task 5: Implement from the flow, the auditable rebooking graph

Build Sofia's machine. Hard rules (identity, policy) are temperature-0 gate agents that call deterministic tools. The diamond into `gate` must wait for **both** upstream nodes, and a failed gate loops back once, capped.

```mermaid
flowchart TD
    V[validate identity] --> E[eligibility]
    E --> RA[reaccom]
    E --> CO[comp]
    RA --> G{policy gate + audit}
    CO --> G
    G -->|fail, capped| RA
    G -->|pass| F[finalize + audit]
```

Cell 5a defines the nodes and the conditions (finish the AND factory). Cell 5b wires the graph (finish the edges and the caps).

In [ ]:
# Cell 5a: nodes + conditions
from strands.multiagent import GraphBuilder
from strands.multiagent.base import Status

g_validate = Agent(model=haiku_t0, name="g_validate",
    system_prompt="Call validate_identity with the record locator and surname from the task. Output EXACTLY 'IDENTITY VERIFIED' or 'IDENTITY DENIED'.",
    tools=[validate_identity])
g_elig = Agent(model=haiku, name="g_elig",
    system_prompt="State whether the disruption is airline-caused (fees waived) and whether a refund is eligible. Use tools.",
    tools=[get_pnr, get_fare_rules, check_refund_eligibility])
g_reaccom = Agent(model=haiku, name="g_reaccom",
    system_prompt="Find alternative flights to the destination. Use tools.", tools=[get_pnr, search_reaccommodation])
g_comp = Agent(model=haiku, name="g_comp",
    system_prompt="Compute duty-of-care compensation for the disruption type and tier. Use tools.",
    tools=[get_pnr, get_loyalty, compute_compensation])
g_gate = Agent(model=haiku_t0, name="g_gate",
    system_prompt="Call apply_policy_gate with the record locator and disruption reason from the task. Call write_audit to log the decision. Output EXACTLY 'POLICY PASS' or 'POLICY FAIL: <reasons>'.",
    tools=[apply_policy_gate, write_audit])
g_final = Agent(model=haiku_t0, name="g_final",
    system_prompt="Call write_audit to record finalization. Then write the final customer message. Warm and concise.",
    tools=[write_audit])

def identity_ok(state):
    r = state.results.get("validate")
    return bool(r) and "identity verified" in str(r.result).lower()

def policy_passed(state):
    r = state.results.get("gate")
    return bool(r) and "policy pass" in str(r.result).lower()

def policy_failed(state):
    r = state.results.get("gate")
    return bool(r) and "policy fail" in str(r.result).lower()

def all_dependencies_complete(required):
    def check(state):
        return all(____ for n in required)    # TODO: node present AND status COMPLETED
    return check

both_ready = all_dependencies_complete(["reaccom", "comp"])
print("nodes and conditions ready")

In [ ]:
# Cell 5b: wire the graph
b = GraphBuilder()
b.add_node(g_validate, "validate")
b.add_node(g_elig,     "eligibility")
b.add_node(g_reaccom,  "reaccom")
b.add_node(g_comp,     "comp")
b.add_node(g_gate,     "gate")
b.add_node(g_final,    "finalize")

b.add_edge("validate", "eligibility", condition=identity_ok)
b.add_edge("eligibility", "reaccom")
b.add_edge("eligibility", "comp")
b.add_edge("reaccom", "gate", condition=____)     # TODO: AND-join (wait for reaccom AND comp)
b.add_edge("comp",    "gate", condition=____)     # TODO: AND-join
b.add_edge("gate", "reaccom",  condition=____)    # TODO: feedback on a failed gate
b.add_edge("gate", "finalize", condition=____)    # TODO: advance on a passed gate

b.set_entry_point("validate")
b.____                    # TODO: cap the executions (covers the feedback loop)
b.____                    # TODO: reset nodes on revisit
rebooking_graph = b.build()

task = ("Involuntary rebooking. PNR JX48Q2, surname Rao. Flight 6E-317 BLR-DEL cancelled by the airline. "
        "Re-accommodate to BOM today. Disruption reason: flight cancelled by airline. Tier: Gold. Type: cancellation.")

AUDIT_LOG.clear()
with meter("t5_rebooking_graph"):
    reply, res = metered_multi(rebooking_graph, task)
print("execution order:", _nodes_run(res))
print("audit log:", AUDIT_LOG)
print("\nreply:\n", reply)

## Task 6: Predict the execution order

Before you trust Task 5, predict its behavior on the clean case.

- The full `execution_order` (five or six node ids, in order): ________
- If you had left the `gate` edges unconditional (no AND-join), and `reaccom` finished before `comp`, what fires the moment `reaccom` completes: ________
- Crash or silent wrong answer: ________

In [ ]:
PREDICTED_ORDER = ["____"]        # TODO: list node ids in order
EARLY_FIRE      = "____"           # TODO: which node fires early without the AND-join
CRASH_OR_SILENT = "____"           # TODO: "crash" or "silent wrong answer"
print(PREDICTED_ORDER, EARLY_FIRE, CRASH_OR_SILENT)

## Task 7: Fill the node-type table

Mark each node **gate** (deterministic hard rule in a tool) or **agent** (model reasoning).

| Node | gate or agent |
|---|---|
| validate identity | ____ |
| eligibility | ____ |
| reaccom | ____ |
| comp | ____ |
| policy gate | ____ |
| finalize + audit | ____ |

Record the two gate-vs-agent calls you were least sure about.

In [ ]:
VALIDATE_TYPE = "____"   # TODO: gate / agent
ELIG_TYPE     = "____"   # TODO: gate / agent
print(VALIDATE_TYPE, ELIG_TYPE)

## Task 8: Complete the flowchart, then wire the composition

The make-good package is open-ended, so put a swarm inside one node of the compliance graph. Fill the blank node, then wire the graph with the swarm as the `options` node.

```mermaid
flowchart TD
    V[validate] --> E[eligibility]
    E --> O[... fill: what kind of node explores the package?]
    O --> G{policy gate}
    G -->|pass| F[finalize]
```

- The `options` node type = ________

In [ ]:
# Inner swarm that explores the make-good package.
c_plan   = Agent(model=haiku, name="package_planner", system_prompt="Assemble the best re-accommodation plus compensation package. Hand off to flight_finder and care_desk, then combine.", tools=[get_pnr])
c_flight = Agent(model=haiku, name="flight_finder",   system_prompt="Find alternative flights. Hand off when done.", tools=[get_pnr, search_reaccommodation])
c_care   = Agent(model=haiku, name="care_desk",       system_prompt="Compute duty-of-care compensation. Hand off to package_planner when done.", tools=[get_pnr, get_loyalty, compute_compensation])

options_swarm = Swarm([c_plan, c_flight, c_care], entry_point=c_plan,
                      max_handoffs=8, max_iterations=8, execution_timeout=400.0, node_timeout=150.0,
                      repetitive_handoff_detection_window=5, repetitive_handoff_min_unique_agents=2)

cb = GraphBuilder()
cb.add_node(g_validate, "validate")
cb.add_node(g_elig,     "eligibility")
cb.add_node(____, "options")                    # TODO: what goes in as the options node?
cb.add_node(g_gate,     "gate")
cb.add_node(g_final,    "finalize")

cb.add_edge("validate", "eligibility", condition=identity_ok)
cb.add_edge("eligibility", "options")
cb.add_edge("options", "gate")
cb.add_edge("gate", "finalize", condition=____)   # TODO: pass condition
cb.set_entry_point("validate")
cb.set_max_node_executions(16)
composed = cb.build()

task9 = ("Involuntary rebooking. PNR JX48Q2, surname Rao. 6E-317 BLR-DEL cancelled by the airline. Destination BOM today. "
         "Disruption reason: flight cancelled by airline. Tier: Gold. Type: cancellation.")
AUDIT_LOG.clear()
with meter("t8_composition"):
    reply9, res9 = metered_multi(composed, task9)
print("outer order:", _nodes_run(res9))
print("audit log:", AUDIT_LOG)
print("\nreply:\n", reply9)

## Task 9: Choose the pattern for three fresh tickets

- **X:** nightly batch re-prices 5,000 fares through the same four fixed steps every time.
- **Y:** VIP escalation, cancelled flight, recovery budget, loyalty exception, legal sensitivity, sequence unknown, must be logged.
- **Z:** refund eligibility on a disputed high-value ticket where a single model pass has been wrong before.

Name the smallest pattern for each.

In [ ]:
# options: chaining, routing, parallelization/voting, orchestrator, evaluator, swarm, graph, composition
PATTERN_X = "____"   # TODO
PATTERN_Y = "____"   # TODO
PATTERN_Z = "____"   # TODO
print(PATTERN_X, "|", PATTERN_Y, "|", PATTERN_Z)

## Task 10: Fix the condition that never fires

This gate never advances to finalize, even on a clean pass. Fix it.

In [ ]:
def broken_policy_passed(state):
    r = state.results.get("policy_gate")       # the node in Task 5 was added as "gate"
    return bool(r) and "policy pass" in str(r.result).lower()

# TODO: write the corrected version below (read the right node id).
def fixed_policy_passed(state):
    r = ____                                   # TODO
    return bool(r) and "policy pass" in str(r.result).lower()

print("fixed:", callable(fixed_policy_passed))

## Task 11: Red-team the composition seam

A teammate ships the inner swarm like this, and points at the outer graph's `set_max_node_executions(16)` as proof the system is bounded.

```python
options_swarm = Swarm([c_plan, c_flight, c_care], entry_point=c_plan)
```

- Is the whole system actually bounded: ________
- The guards missing on the inner swarm: ________
- One line, why the outer graph cap does not save you: ________

In [ ]:
IS_BOUNDED     = "____"   # TODO: yes / no
MISSING_GUARDS = "____"   # TODO: name them
WHY_CAP_FAILS  = "____"   # TODO: one line
print(IS_BOUNDED); print(MISSING_GUARDS); print(WHY_CAP_FAILS)

## Done
You built a swarm with guards, an auditable graph with an AND-join and a capped feedback loop, and a composition with a swarm boxed inside one node. The lesson that outranks all of it: the smallest pattern that solves the problem, bounded on every seam. That is the job.